# Linear Regression: A Practical Guide

This notebook provides a practical walkthrough of the key concepts of Linear Regression, applying them to the `linear-regression-data.csv` dataset. It serves as the hands-on counterpart to the concepts explained in the `linear-regression.md` document.

## 0. Generating the Dataset

First, we'll create the `linear_regression_data.csv` file ourselves. This step ensures that the notebook is self-contained and can be run without needing to download any external files. We'll generate a simple dataset where 'Test Score' has a clear positive linear relationship with 'Hours Studied', plus some random noise to make it more realistic.

In [1]:
import pandas as pd
import numpy as np

# --- Generate and Save Synthetic Data ---

# Set a seed for reproducibility, so the results are the same every time
np.random.seed(42)

# Generate 'Hours Studied' data: 50 data points between 0 and 10
hours_studied = np.random.rand(50, 1) * 10

# Generate 'Test Score' data with a linear relationship and some noise
# Formula: score = 50 (base) + 5 * hours + noise
noise = np.random.randn(50, 1) * 5
test_score = 50 + (5 * hours_studied) + noise

# Create a DataFrame
data = pd.DataFrame({
    'Hours Studied': hours_studied.flatten(),
    'Test Score': test_score.flatten()
})

# Ensure test scores are within a plausible range (e.g., 0 to 100)
data['Test Score'] = data['Test Score'].clip(0, 100)

# Save the DataFrame to a CSV file, which the rest of the notebook will use
data.to_csv('linear-regression-data.csv', index=False)

print("File 'linear-regression-data.csv' has been generated.")
print("--- First 5 rows of the generated data ---")
print(data.head())

## 1. Loading and Exploring the Data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset we just created
df = pd.read_csv('linear-regression-data.csv')

# --- Basic Information ---
print("--- Displaying the first 5 rows ---")
print(df.head())

print("\n--- Displaying a summary of the DataFrame ---")
df.info()

print("\n--- Displaying descriptive statistics ---")
print(df.describe())

## 2. Visualizing the Data
A scatter plot is a great way to visualize the relationship between two variables.

In [3]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Hours Studied', y='Test Score', data=df)
plt.title('Hours Studied vs. Test Score')
plt.xlabel('Hours Studied')
plt.ylabel('Test Score')
plt.grid(True)
plt.show()

## 3. Manual Calculation of Linear Regression Parameters
We'll calculate the slope (m) and intercept (c) using the formulas from `Linear Regression.md`.

In [4]:
x = df['Hours Studied']
y = df['Test Score']

# Calculate means
x_mean = np.mean(x)
y_mean = np.mean(y)

# Calculate slope (m)
numerator = np.sum((x - x_mean) * (y - y_mean))
denominator = np.sum((x - x_mean)**2)
m = numerator / denominator

# Calculate intercept (c)
c = y_mean - m * x_mean

print(f"Calculated Slope (m): {m}")
print(f"Calculated Intercept (c): {c}")

## 4. Making Predictions

In [5]:
df['Predicted Score (Manual)'] = m * df['Hours Studied'] + c
print(df.head())

## 5. Evaluating Model Performance

In [6]:
from sklearn.metrics import mean_squared_error, r2_score

y_actual = df['Test Score']
y_predicted_manual = df['Predicted Score (Manual)']

# Mean Squared Error (MSE)
mse_manual = mean_squared_error(y_actual, y_predicted_manual)
print(f"Mean Squared Error (MSE): {mse_manual}")

# Root Mean Squared Error (RMSE)
rmse_manual = np.sqrt(mse_manual)
print(f"Root Mean Squared Error (RMSE): {rmse_manual}")

# R-squared (R²)
r2_manual = r2_score(y_actual, y_predicted_manual)
print(f"R-squared (R²): {r2_manual}")

## 6. Using scikit-learn for Linear Regression

In [7]:
from sklearn.linear_model import LinearRegression

# Prepare the data for scikit-learn
X = df[['Hours Studied']] # Features need to be 2D
y = df['Test Score'] # Target

# Create and train the model
model = LinearRegression()
model.fit(X, y)

# Get parameters
m_sklearn = model.coef_[0]
c_sklearn = model.intercept_

print(f"scikit-learn Slope (m): {m_sklearn}")
print(f"scikit-learn Intercept (c): {c_sklearn}")

## 7. Visualizing the Results

In [8]:
plt.figure(figsize=(12, 7))
sns.scatterplot(x='Hours Studied', y='Test Score', data=df, label='Actual Data')
plt.plot(df['Hours Studied'], df['Predicted Score (Manual)'], color='red', linewidth=2, label='Regression Line')
plt.title('Hours Studied vs. Test Score with Regression Line')
plt.xlabel('Hours Studied')
plt.ylabel('Test Score')
plt.legend()
plt.grid(True)
plt.show()

## 8. Checking the Assumptions of Linear Regression

As mentioned in the markdown, a good linear regression model should satisfy certain assumptions. We can check some of these by analyzing the *residuals* (the difference between actual and predicted values).

In [9]:
import scipy.stats as stats

# Calculate residuals
residuals = df['Test Score'] - df['Predicted Score (Manual)']

# 1. Residual Plot (for Homoscedasticity)
# This plot shows if the variance of residuals is constant.
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.scatterplot(x=df['Predicted Score (Manual)'], y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Residual Plot')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')

# 2. Histogram of Residuals (for Normality)
# This shows if the residuals are normally distributed.
plt.subplot(1, 3, 2)
sns.histplot(residuals, kde=True)
plt.title('Histogram of Residuals')
plt.xlabel('Residuals')

# 3. Q-Q Plot (for Normality)
# This is another way to check for normality. Points should follow the red line.
plt.subplot(1, 3, 3)
stats.probplot( residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

## 9. (Optional) Implementing Linear Regression with Gradient Descent

Instead of calculating the slope and intercept directly, we can use an optimization algorithm like Gradient Descent to find them. This is an iterative approach that is especially useful for more complex models.

In [10]:
# --- Gradient Descent Implementation ---

# Parameters
m_gd = 0.0
c_gd = 0.0
learning_rate = 0.001
epochs = 10000

# Number of data points
n = float(len(x))

# Performing Gradient Descent
for i in range(epochs):
    y_pred = m_gd * x + c_gd
    D_m = (-2/n) * sum(x * (y - y_pred)) # Derivative wrt m
    D_c = (-2/n) * sum(y - y_pred)      # Derivative wrt c
    m_gd = m_gd - learning_rate * D_m   # Update m
    c_gd = c_gd - learning_rate * D_c   # Update c

print(f"Gradient Descent Slope (m): {m_gd}")
print(f"Gradient Descent Intercept (c): {c_gd}")

You can see that the values for slope and intercept found via Gradient Descent are very close to the ones we calculated manually and with scikit-learn. The small differences are due to the iterative nature of the algorithm and the choice of learning rate and epochs.

## 10. Multiple Linear Regression

Simple linear regression uses one feature to predict a target. **Multiple Linear Regression** extends this by using multiple features. The goal is the same, but the equation now includes a separate coefficient (slope) for each feature.

Equation: `y = m1*x1 + m2*x2 + ... + c`

Each coefficient (`m1`, `m2`, etc.) represents the change in `y` for a one-unit change in that feature, holding all other features constant. We'll add a new feature, 'Hours Slept', to our dataset to see how this works in practice.

In [ ]:
# --- Generate a new feature 'Hours Slept' ---
np.random.seed(42) # Reset seed for consistency
hours_slept = np.random.uniform(4, 9, 50) # Uniformly distributed between 4 and 9 hours
df['Hours Slept'] = hours_slept

# Update the 'Test Score' to also depend on 'Hours Slept'
# Formula: score = 50 + 4*studied + 2*slept + noise
noise = np.random.randn(50) * 4
df['Test Score'] = 50 + (4 * df['Hours Studied']) + (2 * df['Hours Slept']) + noise
df['Test Score'] = df['Test Score'].clip(0, 100)

print("--- Data with 'Hours Slept' feature ---")
print(df.head())

# --- Train a Multiple Linear Regression Model ---
from sklearn.linear_model import LinearRegression

# Features are now 'Hours Studied' and 'Hours Slept'
X_multi = df[['Hours Studied', 'Hours Slept']]
y_multi = df['Test Score']

multi_model = LinearRegression()
multi_model.fit(X_multi, y_multi)

# Get the coefficients (m1, m2) and the intercept (c)
m_multi = multi_model.coef_
c_multi = multi_model.intercept_

print(f"\nCoefficients (m1, m2): {m_multi}")
print(f"Intercept (c): {c_multi}")

# --- Evaluate the Multiple Regression Model ---
from sklearn.metrics import r2_score

y_pred_multi = multi_model.predict(X_multi)
r2_multi = r2_score(y_multi, y_pred_multi)

print(f"\nR-squared of the multiple regression model: {r2_multi}")